<a href="https://colab.research.google.com/github/iago7almeida/Data_Analysis/blob/main/tratamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [60]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, sum as spark_sum, year, date_format
from pyspark.sql.types import FloatType, DataType, IntegerType

spark = SparkSession.builder.appName("Processing_data").getOrCreate()

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
#
df_video = spark.read.csv("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/videos-stats.csv", header=True, inferSchema=True)
df_comentario = spark.read.csv("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/comments.csv", header=True, inferSchema=True)

df_video.printSchema()
df_comentario.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: double (nullable = true)
 |-- Comments: double (nullable = true)
 |-- Views: double (nullable = true)

root
 |-- _c0: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [36]:
df_video = df_video.fillna({'Likes': 0, 'Comments': 0, 'Views': 0})

In [37]:
print('Quantidade de registros em df_video: ', df_video.count())
print('Quantidade de registros em df_comentario: ', df_comentario.count())

Quantidade de registros em df_video:  1881
Quantidade de registros em df_comentario:  30036


In [38]:
df_video = df_video.na.drop(subset=["Video ID"])
df_comentario = df_comentario.na.drop(subset=["Video ID"])

print('Quantidade de registros em df_video: ', df_video.count())
print('Quantidade de registros em df_comentario: ', df_comentario.count())

Quantidade de registros em df_video:  1881
Quantidade de registros em df_comentario:  22555


In [39]:
df_video = df_video.dropDuplicates(['Video ID'])
print('Quantidade de registros em df_video: ', df_video.count())

Quantidade de registros em df_video:  1869


In [41]:
df_video = df_video.withColumn("Likes", col("Likes").cast(IntegerType())) \
                   .withColumn("Comments", col("Comments").cast(IntegerType())) \
                   .withColumn("Views", col("Views").cast(IntegerType()))

In [42]:
df_comentario = df_comentario.withColumn("Likes", col("Likes").cast(IntegerType())) \
                             .withColumn("Sentiment", col("Sentiment").cast(IntegerType())) \
                             .withColumnRenamed("Likes", "Likes Comment")

In [43]:
df_video = df_video.withColumn("Interaction", col("Likes") + col("Comments") + col("Views"))

df_video.show(5)

+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+
| _c0|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|
+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+
| 986|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|
|  71|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|
|  48|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|
| 993|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|mukbang| 45628|   17264| 5283664|    5346556|
|1456|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|physics| 10959|     525|  844015|     855499|
+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+
only showing top 5 rows



In [48]:
#tranformar dados para date
df_video = df_video.withColumn("Published At", col("Published At").cast("date"))
print(df_video.printSchema())

root
 |-- _c0: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Video ID: string (nullable = true)
 |-- Published At: date (nullable = true)
 |-- Keyword: string (nullable = true)
 |-- Likes: integer (nullable = true)
 |-- Comments: integer (nullable = true)
 |-- Views: integer (nullable = true)
 |-- Interaction: integer (nullable = true)

None


In [49]:
df_video = df_video.withColumn("Year", year(col("Published At")))
df_video.show(5)

+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
| _c0|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|
+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
| 986|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|
|  71|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|
|  48|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|2022|
| 993|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|mukbang| 45628|   17264| 5283664|    5346556|2022|
|1456|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|physics| 10959|     525|  844015|     855499|2017|
+----+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
only showing top 5 rows



In [51]:
df_join_video_coments = df_video.join(df_comentario, df_video["Video ID"] == df_comentario["Video ID"])
df_join_video_coments.show(5)

df_join2 = df_video.join(df_comentario, on="Video ID") # inclui o "Video ID"
df_join2.show(5)

+---+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+---+-----------+--------------------+-------------+---------+
|_c0|               Title|   Video ID|Published At|Keyword|Likes|Comments| Views|Interaction|Year|_c0|   Video ID|             Comment|Likes Comment|Sentiment|
+---+--------------------+-----------+------------+-------+-----+--------+------+-----------+----+---+-----------+--------------------+-------------+---------+
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  0|wAZZ-UWGVHI|Let's not forget ...|           95|        1|
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  1|wAZZ-UWGVHI|Here in NZ 50% of...|           19|        0|
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  2022-08-23|   tech| 3407|     672|135612|     139691|2022|  2|wAZZ-UWGVHI|I will forever ac...|          161|        2|
|  0|Apple Pay Is Kill...|wAZZ-UWGVHI|  

In [52]:
df_us_videos = spark.read.csv("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/USvideos.csv", header=True, inferSchema=True)
df_us_videos.show(5)

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [54]:
df_join_video_usvideos = df_us_videos.dropDuplicates(['video_id']) # Remover os valores duplicados em us videos
df_join_video_usvideos = df_video.join(df_us_videos, on="Title")
df_join_video_usvideos.show(5)

+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|               Title|_c0|   Video ID|Published At|Keyword| Likes|Comments|    Views|Interaction|Year|   video_id|trending_date|channel_title|category_id|        publish_time|                tags|   views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+--------------------+---+-----------+------------+-------+------+--------+---------+-----------+----+-----------+-------------+-------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------

In [55]:
from pyspark.sql.functions import isnan, count
df_video.select([count(when(col(c).isNull(), c)).alias(c) for c in df_video.columns]).show()

+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|_c0|Title|Video ID|Published At|Keyword|Likes|Comments|Views|Interaction|Year|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+
|  0|    0|       0|           0|      0|    0|       0|    0|          0|   0|
+---+-----+--------+------------+-------+-----+--------+-----+-----------+----+



In [56]:
df_video = df_video.drop("_c0")
df_video.write.mode("overwrite").option("header", "true").parquet("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/df_video.parquet")

In [57]:
df_join2 = df_join2.drop("_c0")
df_join2.write.mode("overwrite").option("header", "true").parquet("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/df_join2.parquet")

In [62]:
spark.read.parquet("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/df_video.parquet").show(5)
spark.read.parquet("/content/drive/My Drive/Colab Notebooks/Data Processing - PySpark/df_join2.parquet").show(5)

+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
|               Title|   Video ID|Published At|Keyword| Likes|Comments|   Views|Interaction|Year|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|mukbang|378858|   18860|17975269|   18372987|2020|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|   news|  6379|    4853|  808787|     820019|2022|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|   news|  1029|    2347|   97434|     100810|2022|
|Celebrating My 40...|-64r1hcxtV4|  2022-05-30|mukbang| 45628|   17264| 5283664|    5346556|2022|
|Physics Review - ...|-6IgkG5yZfo|  2017-01-02|physics| 10959|     525|  844015|     855499|2017|
+--------------------+-----------+------------+-------+------+--------+--------+-----------+----+
only showing top 5 rows

+-----------+--------------------+------------+-------+-----+--------+------+-----------+----

In [63]:
spark.stop()